
# Roman ray tracing: analytic optical surfaces in PCS coordinates

This notebook refactors the earlier `romanraytracingplots.ipynb` workflow into three layers:

1. **Trace rays** with `_RomanRayBundle(..., save_trace_history=True)`.
2. **Construct optical surfaces analytically** from the same `Rinv`, `K`, transforms, and `activeZone` descriptions used by `_RomanRayBundle`.
3. **Transform each surface from its local coordinates into PCS coordinates** with `RomanRayBundle.MV`, then overlay the surfaces and saved ray trajectories.

The conic sag is

$$ z(x,y)=\frac{R_{\rm inv}(x^{2} + y^{2})}{1 + \sqrt{1-(1+K)R_{\rm inv}^{2} (x^{2} + y^{2})}} $$

All physical positions are in **mm**.

### Trace-history order used here

For the normal (`ghostpath=False`) path represented by the current 9 saved locations:

| Trace index | Location |
|---:|---|
| 0 | entrance/start plane |
| 1 | M1 primary |
| 2 | M2 secondary |
| 3 | FM1 |
| 4 | FM2 |
| 5 | M3 tertiary |
| 6 | filter S1 |
| 7 | filter S2 |
| 8 | FPA |

The validation section checks this mapping numerically.


In [ ]:
%matplotlib widget

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection


import psfsim.romantrace as rt
from psfsim.romantrace import _RomanRayBundle
from psfsim.offsets import sm_offset, fpa_offset, fbias_offset

# Use the same transform builder that romantrace.py uses.
build_transform_matrix = rt.build_transform_matrix



## Make a dense ray bundle and recover the saved trace

`N=100` gives a 100 × 100 entrance-pupil sampling, i.e. up to 10,000 rays.
For faster interactive iteration, temporarily reduce `RAYBUNDLE_SIDELENGTH`.


In [ ]:
XAN = 0.3
YAN = 0.1
RAYBUNDLE_SIDELENGTH = 100
FILTER = "J"

IDEAL_GEOM = False
GHOSTPATH = False

rrb = _RomanRayBundle(
    XAN,
    YAN,
    RAYBUNDLE_SIDELENGTH,
    FILTER,
    idealgeom=IDEAL_GEOM,
    ghostpath=GHOSTPATH,
    save_trace_history=True,
)

history = np.asarray(rrb.trace_history, dtype=float)
number_of_intersections = history.shape[0]

# (n_saved_locations, N, N, 3) -> (n_saved_locations, N**2, 3)
rays = history.reshape(number_of_intersections, -1, 3)
number_of_rays = rays.shape[1]

print("history shape:", history.shape)
print("rays shape:   ", rays.shape)
print("saved locations:", number_of_intersections)
print("rays:", number_of_rays)


In [ ]:
N_RAYS_TO_PLOT = 36

open_mask = np.asarray(rrb.open, dtype=bool).reshape(-1)
open_ray_indices = np.flatnonzero(open_mask)

number_of_open_rays = len(open_ray_indices)
number_of_blocked_rays = number_of_rays - number_of_open_rays

print(f"Total rays:             {number_of_rays}")
print(f"Reach detector:         {number_of_open_rays}")
print(f"Blocked somewhere:      {number_of_blocked_rays}")
print(
    f"Throughput:             "
    f"{100 * number_of_open_rays / number_of_rays:.2f}%"
)

if number_of_open_rays == 0:
    ray_indices = np.array([], dtype=int)

else:
    n_plot = min(
        N_RAYS_TO_PLOT,
        number_of_open_rays,
    )

    sample_positions = np.linspace(
        0,
        number_of_open_rays - 1,
        n_plot,
        dtype=int,
    )

    ray_indices = open_ray_indices[sample_positions]

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

for ray in ray_indices:
    ax.plot(
        rays[:, ray, 0],
        rays[:, ray, 1],
        rays[:, ray, 2],
        marker="o",
        markersize=2,
    )

ax.set_xlabel("x [mm]")
ax.set_ylabel("y [mm]")
ax.set_zlabel("z [mm]")
plt.show()



## Geometry helpers

### Homogeneous coordinates

RayBundle positions are homogeneous points `[1, x, y, z]`. The matrix returned by
`build_transform_matrix(...)` therefore carries the complete local-surface → PCS transform,
including translation.

The helpers below construct homogeneous points and use `rrb.MV(...)` directly. No need to
add a separate surface origin afterward.


In [ ]:
def xyz_to_homogeneous(xyz):
    """Convert (..., 3) Cartesian positions to (..., 4) points [1,x,y,z]."""
    xyz = np.asarray(xyz, dtype=float)

    if xyz.shape[-1] != 3:
        raise ValueError("Expected xyz with final dimension 3.")

    h = np.ones(xyz.shape[:-1] + (4,), dtype=float)
    h[..., 1:4] = xyz
    return h


def transform_xyz_with_mv(rb, transform, xyz):
    """
    Transform Cartesian positions with RayBundle.MV.

    Points are temporarily reshaped to (-1, 1, 4), matching the two-dimensional
    pupil-grid style used internally by RayBundle.
    """
    xyz = np.asarray(xyz, dtype=float)
    original_shape = xyz.shape

    h = xyz_to_homogeneous(xyz).reshape(-1, 1, 4)
    transformed_h = rb.MV(transform, h)

    transformed_xyz = np.asarray(transformed_h)[:, 0, 1:4]
    return transformed_xyz.reshape(original_shape)


def surface_to_pcs(rb, transform, local_xyz):
    """Surface-local Cartesian coordinates -> PCS Cartesian coordinates."""
    return transform_xyz_with_mv(rb, transform, local_xyz)


def pcs_to_surface(rb, transform, pcs_xyz):
    """PCS Cartesian coordinates -> surface-local Cartesian coordinates."""
    return transform_xyz_with_mv(rb, np.linalg.inv(transform), pcs_xyz)



### Conic sag

For `Rinv = 0`, the same expression gives `z = 0`, so the helper also covers planar fold mirrors.


In [ ]:
def conic_sag(x, y, Rinv, K):
    """
    Conic sag in surface-local coordinates.

    x, y : mm
    Rinv : 1/mm
    K    : dimensionless
    z    : mm
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    r2 = x**2 + y**2
    discriminant = 1.0 - (1.0 + K) * (Rinv**2) * r2

    z = np.full(np.broadcast(x, y).shape, np.nan, dtype=float)
    valid = discriminant >= 0.0

    z[valid] = (
        Rinv * r2[valid]
        / (1.0 + np.sqrt(discriminant[valid]))
    )

    return z



### Plotting version of the relevant `activeZone` tests

For M1/M2/FM1/FM2/M3/S1/S2, the supplied surface definitions only require the geometric keys below:

- `CIR`: circular outer region
- `OBS`: circular central obstruction
- `REX`, `REY`: rectangular half-extents
- `ADX`, `ADY`: decenter
- `ARO`: aperture rotation (supported here for completeness)

Each dictionary is one primitive region. Multiple dictionaries are combined as a **union**.
Inside one dictionary, constraints are combined; for example, M1's `CIR` + `OBS` becomes an annulus.

This is deliberately a plotting helper, not a rewrite of `RayBundle.activeZone`.
If we later want to draw baffles/stops containing `HOL`, `iREX`, `iREY`, `iCIR_ORIG`, etc.,
we'll need to copy those remaining tests directly from `RayBundle.activeZone`.


In [ ]:
def _one_active_zone_mask(x, y, zone):
    """Boolean mask for one activeZone primitive in surface-local coordinates."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    adx = float(zone.get("ADX", 0.0))
    ady = float(zone.get("ADY", 0.0))
    angle = np.deg2rad(float(zone.get("ARO", 0.0)))

    dx = x - adx
    dy = y - ady

    # Inverse-rotate sample points into the primitive's own frame.
    c = np.cos(angle)
    s = np.sin(angle)
    xr = c * dx + s * dy
    yr = -s * dx + c * dy

    mask = np.ones(np.broadcast(x, y).shape, dtype=bool)
    has_positive_shape = False

    if "CIR" in zone:
        radius = float(zone["CIR"])
        mask &= (xr**2 + yr**2) <= radius**2
        has_positive_shape = True

    if "REX" in zone:
        mask &= np.abs(xr) <= float(zone["REX"])
        has_positive_shape = True

    if "REY" in zone:
        mask &= np.abs(yr) <= float(zone["REY"])
        has_positive_shape = True

    if "OBS" in zone:
        radius = float(zone["OBS"])
        mask &= (xr**2 + yr**2) >= radius**2

    if not has_positive_shape:
        raise NotImplementedError(
            f"Plotting helper does not yet know the positive shape in zone: {zone}"
        )

    return mask


def active_zone_mask(x, y, active_zone):
    """Union the activeZone primitive dictionaries into one physical-surface mask."""
    if active_zone is None:
        return np.ones(np.broadcast(x, y).shape, dtype=bool)

    mask = np.zeros(np.broadcast(x, y).shape, dtype=bool)

    for zone in active_zone:
        mask |= _one_active_zone_mask(x, y, zone)

    return mask



### Infer local plotting limits from the `activeZone`

This keeps the surface construction generic instead of hard-coding a separate x/y box for every mirror.


In [ ]:
def infer_active_zone_bounds(active_zone, padding_fraction=0.05):
    """Infer a rectangular local plotting extent from CIR / REX / REY primitives."""
    if active_zone is None:
        raise ValueError("Give explicit limits when active_zone is None.")

    xmin = np.inf
    xmax = -np.inf
    ymin = np.inf
    ymax = -np.inf

    for zone in active_zone:
        adx = float(zone.get("ADX", 0.0))
        ady = float(zone.get("ADY", 0.0))
        theta = np.deg2rad(float(zone.get("ARO", 0.0)))

        if "CIR" in zone:
            rx = ry = float(zone["CIR"])
        elif "REX" in zone or "REY" in zone:
            rx = float(zone.get("REX", 0.0))
            ry = float(zone.get("REY", 0.0))
        else:
            continue

        dx = abs(np.cos(theta)) * rx + abs(np.sin(theta)) * ry
        dy = abs(np.sin(theta)) * rx + abs(np.cos(theta)) * ry

        xmin = min(xmin, adx - dx)
        xmax = max(xmax, adx + dx)
        ymin = min(ymin, ady - dy)
        ymax = max(ymax, ady + dy)

    if not np.all(np.isfinite([xmin, xmax, ymin, ymax])):
        raise ValueError("Could not infer limits from activeZone.")

    span = max(xmax - xmin, ymax - ymin)
    pad = padding_fraction * span

    return (xmin - pad, xmax + pad), (ymin - pad, ymax + pad)



## Surface specifications copied from `_RomanRayBundle`

The transforms, `Rinv`, `K`, and active zones below are the same values used by the trace.
For M2, non-ideal geometry includes `sm_offset["DZ"]`, matching `_RomanRayBundle`.


In [ ]:
def build_optical_surface_specs(idealgeom=False):
    """Build the same local-to-PCS transforms and conic constants used by _RomanRayBundle."""
    sm_offset_use = 0.0 if idealgeom else sm_offset["DZ"]

    TF2 = build_transform_matrix(
        xde=466.3656874216886,
        yde=-807.7690655211503,
        zde=-387.2147203297053,
        ade=100.0982115641859,
        bde=29.61417435444774,
        cde=174.9705371700583,
    )

    return {
        "M1": {
            "transform": build_transform_matrix(zde=660.4, ade=-180, cde=180),
            "Rinv": -1.0 / 5671.1342,
            "K": -0.9728630311,
            "activeZone": [{"CIR": 1184.02, "OBS": 321.31}],
            "trace_index": 1,
        },

        "M2": {
            "transform": build_transform_matrix(
                zde=2945.4 + sm_offset_use,
                ade=-180,
                cde=180,
            ),
            "Rinv": -1.0 / 1299.6164,
            "K": -1.6338521231,
            "activeZone": [{"CIR": 266.255}],
            "trace_index": 2,
        },

        "FM1": {
            "transform": build_transform_matrix(
                xde=-73.371025,
                yde=127.0823431034063,
                zde=-299.6,
                ade=135.6742566218209,
                bde=21.96993018862709,
                cde=159.0416363940703,
            ),
            "Rinv": 0.0,
            "K": 0.0,
            "activeZone": [
                {"REX": 134.13, "REY": 152.42, "ADY": 28.84},
                {"REX": 151.11, "REY": 135.44, "ADY": 28.84},
                {"CIR": 16.98, "ADX": -134.13, "ADY": 164.28},
                {"CIR": 16.98, "ADX": 134.13, "ADY": 164.28},
                {"CIR": 16.98, "ADX": 134.13, "ADY": -106.6},
                {"CIR": 16.98, "ADX": -134.13, "ADY": -106.6},
            ],
            "trace_index": 3,
        },

        "FM2": {
            "transform": TF2,
            "Rinv": 0.0,
            "K": 0.0,
            "activeZone": [
                {"CIR": 47.7, "ADX": 169.24, "ADY": -92.27},
                {"CIR": 47.7, "ADX": -169.24, "ADY": -92.27},
                {"REX": 169.255, "REY": 47.7, "ADY": -92.27},
                {"REX": 216.955, "REY": 142.135, "ADY": 49.865},
            ],
            "trace_index": 4,
        },

        "M3": {
            "transform": build_transform_matrix(
                xde=85.50941274300123,
                yde=-148.1066473962562,
                zde=-938.9487447474822,
                ade=117.6411883903639,
                bde=27.08781188751067,
                cde=166.5871249774839,
            ),
            "Rinv": -1.0 / 1643.2784,
            "K": -0.5965290831,
            "activeZone": [
                {"REX": 197.435, "REY": 207.00565, "ADY": 222.29065},
                {"REX": 302.715, "REY": 101.72565, "ADY": 222.29065},
                {"CIR": 105.28, "ADX": -197.435, "ADY": 324.0163},
                {"CIR": 105.28, "ADX": 197.435, "ADY": 324.0163},
                {"CIR": 105.28, "ADX": 197.435, "ADY": 120.565},
                {"CIR": 105.28, "ADX": -197.435, "ADY": 120.565},
                {"REX": 256.189698, "REY": 31.9859, "ADY": 444.7891},
            ],
            "trace_index": 5,
        },

        "S1": {
            "transform": build_transform_matrix(
                xde=531.5125171153529,
                yde=-920.606684502614,
                zde=-539.1713886876893,
                ade=102.76851389522,
                bde=29.38268469198068,
                cde=173.6554980927907,
            ),
            "Rinv": -1.0 / 1500.0,
            "K": 0.0,
            "activeZone": [{"CIR": 52.65}],
            "trace_index": 6,
        },

        "S2": {
            "transform": build_transform_matrix(
                xde=536.4189215396419,
                yde=-929.1048262479633,
                zde=-537.2455687321163,
                ade=102.76851389522,
                bde=29.38268469198068,
                cde=173.6554980927907,
            ),
            "Rinv": -1.0 / 1499.31453814,
            "K": 0.0,
            "activeZone": [{"CIR": 52.65}],
            "trace_index": 7,
        },
    }


surface_specs = build_optical_surface_specs(idealgeom=IDEAL_GEOM)
surface_specs.keys()



## `make_conic_surface_pcs`

This is the main reusable function. It:

1. makes a local `(x,y)` mesh,
2. evaluates the exact conic sag,
3. removes the analytic continuation outside the physical `activeZone`,
4. builds local `[1,x,y,z]` points,
5. transforms them with `rrb.MV`,
6. returns both local and PCS grids.


In [ ]:
def make_conic_surface_pcs(
    rb,
    spec,
    ngrid=180,
    xlim=None,
    ylim=None,
):
    """
    Construct one optical surface and transform it from local coordinates to PCS.
    """
    if xlim is None or ylim is None:
        inferred_xlim, inferred_ylim = infer_active_zone_bounds(spec["activeZone"])

        if xlim is None:
            xlim = inferred_xlim
        if ylim is None:
            ylim = inferred_ylim

    x = np.linspace(xlim[0], xlim[1], ngrid)
    y = np.linspace(ylim[0], ylim[1], ngrid)
    X, Y = np.meshgrid(x, y)

    Z = conic_sag(X, Y, spec["Rinv"], spec["K"])

    valid = (
        np.isfinite(Z)
        & active_zone_mask(X, Y, spec["activeZone"])
    )

    local_xyz = np.stack([X, Y, Z], axis=-1)

    pcs_xyz = surface_to_pcs(
        rb,
        spec["transform"],
        local_xyz,
    )

    local_xyz = local_xyz.copy()
    pcs_xyz = pcs_xyz.copy()

    local_xyz[~valid] = np.nan
    pcs_xyz[~valid] = np.nan

    return {
        "X_local": local_xyz[..., 0],
        "Y_local": local_xyz[..., 1],
        "Z_local": local_xyz[..., 2],
        "X_pcs": pcs_xyz[..., 0],
        "Y_pcs": pcs_xyz[..., 1],
        "Z_pcs": pcs_xyz[..., 2],
        "local_xyz": local_xyz,
        "pcs_xyz": pcs_xyz,
        "valid": valid,
        "xlim": xlim,
        "ylim": ylim,
    }


In [ ]:
m1_surface = make_conic_surface_pcs(
    rrb,
    surface_specs["M1"],
    ngrid=220,
)

print("M1 local shape:", m1_surface["local_xyz"].shape)
print("M1 PCS shape:  ", m1_surface["pcs_xyz"].shape)


In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

ax.plot_surface(
    m1_surface["X_pcs"],
    m1_surface["Y_pcs"],
    m1_surface["Z_pcs"],
    linewidth=0,
    alpha=0.65,
)

ax.set_xlabel("PCS x [mm]")
ax.set_ylabel("PCS y [mm]")
ax.set_zlabel("PCS z [mm]")
ax.set_title("Analytic M1 primary mirror in PCS coordinates")

plt.show()



## Construct all main reflecting/refracting surfaces

For a first telescope diagram:

`M1 -> M2 -> FM1 -> FM2 -> M3 -> S1 -> S2`

Use a moderate mesh density during iteration and increase it for the final figure.


In [ ]:
SURFACE_NGRID = 160

surfaces_pcs = {
    name: make_conic_surface_pcs(
        rrb,
        spec,
        ngrid=SURFACE_NGRID,
    )
    for name, spec in surface_specs.items()
}

list(surfaces_pcs)


In [ ]:
surface_rows = []

for name, surface in surfaces_pcs.items():
    local = surface["local_xyz"].reshape(-1, 3)
    pcs = surface["pcs_xyz"].reshape(-1, 3)

    good = np.all(np.isfinite(pcs), axis=1)
    local = local[good]
    pcs = pcs[good]

    surface_rows.append({
        "surface": name,
        "n_grid_points": len(pcs),
        "local_x_min_mm": np.min(local[:, 0]),
        "local_x_max_mm": np.max(local[:, 0]),
        "local_y_min_mm": np.min(local[:, 1]),
        "local_y_max_mm": np.max(local[:, 1]),
        "local_z_min_mm": np.min(local[:, 2]),
        "local_z_max_mm": np.max(local[:, 2]),
        "pcs_x_mean_mm": np.mean(pcs[:, 0]),
        "pcs_y_mean_mm": np.mean(pcs[:, 1]),
        "pcs_z_mean_mm": np.mean(pcs[:, 2]),
    })

surface_summary = pd.DataFrame(surface_rows)
surface_summary



## Validate each saved ray intersection against its analytic surface

The check is performed in each surface's own local coordinate system:

\[
\Delta z =
z_{\rm ray,local} -
z_{\rm conic}(x_{\rm local},y_{\rm local}).
\]

Small residuals verify the trace index, transform, `Rinv`, and `K` together.


In [ ]:
def validate_trace_surface(rb, history, name, spec):
    trace_index = spec["trace_index"]

    hits_pcs = np.asarray(history[trace_index], dtype=float).reshape(-1, 3)
    finite = np.all(np.isfinite(hits_pcs), axis=1)
    hits_pcs = hits_pcs[finite]

    hits_local = pcs_to_surface(
        rb,
        spec["transform"],
        hits_pcs,
    )

    x_local = hits_local[:, 0]
    y_local = hits_local[:, 1]
    z_local = hits_local[:, 2]

    z_expected = conic_sag(
        x_local,
        y_local,
        spec["Rinv"],
        spec["K"],
    )

    residual = z_local - z_expected

    in_active_zone = active_zone_mask(
        x_local,
        y_local,
        spec["activeZone"],
    )

    use = np.isfinite(residual) & in_active_zone

    if np.any(use):
        rmse = np.sqrt(np.mean(residual[use]**2))
        mae = np.mean(np.abs(residual[use]))
        max_abs = np.max(np.abs(residual[use]))
    else:
        rmse = np.nan
        mae = np.nan
        max_abs = np.nan

    return {
        "surface": name,
        "trace_index": trace_index,
        "n_finite_hits": len(hits_local),
        "fraction_in_active_zone": np.mean(in_active_zone) if len(in_active_zone) else np.nan,
        "local_z_residual_rmse_mm": rmse,
        "local_z_residual_mae_mm": mae,
        "local_z_residual_maxabs_mm": max_abs,
    }


validation_df = pd.DataFrame([
    validate_trace_surface(
        rrb,
        history,
        name,
        spec,
    )
    for name, spec in surface_specs.items()
])

validation_df



If one surface has a much larger residual than the others, inspect that surface before changing the conic equation. Likely causes include a trace-index mismatch, different `idealgeom` setting, a ghost path, or an `activeZone` test that needs to be ported more literally.



## Primary-mirror statistics in M1's local frame

This refactors the earlier primary point-cloud analysis so curvature is measured in the coordinate system where the conic prescription is actually defined.


In [ ]:
m1_spec = surface_specs["M1"]

m1_hits_pcs = history[m1_spec["trace_index"]].reshape(-1, 3)
good = np.all(np.isfinite(m1_hits_pcs), axis=1)
m1_hits_pcs = m1_hits_pcs[good]

m1_hits_local = pcs_to_surface(
    rrb,
    m1_spec["transform"],
    m1_hits_pcs,
)

m1_df = pd.DataFrame(
    m1_hits_local,
    columns=["x_local_mm", "y_local_mm", "z_local_mm"],
)

m1_df["r_local_mm"] = np.sqrt(
    m1_df["x_local_mm"]**2
    + m1_df["y_local_mm"]**2
)

m1_df["z_conic_mm"] = conic_sag(
    m1_df["x_local_mm"].to_numpy(),
    m1_df["y_local_mm"].to_numpy(),
    m1_spec["Rinv"],
    m1_spec["K"],
)

m1_df["z_residual_mm"] = (
    m1_df["z_local_mm"]
    - m1_df["z_conic_mm"]
)

m1_df.describe()


In [ ]:
order = np.argsort(m1_df["r_local_mm"].to_numpy())

plt.figure(figsize=(9, 6))

plt.scatter(
    m1_df["r_local_mm"],
    m1_df["z_local_mm"],
    s=3,
    alpha=0.25,
    label="Ray intersections",
)

plt.plot(
    m1_df["r_local_mm"].to_numpy()[order],
    m1_df["z_conic_mm"].to_numpy()[order],
    linewidth=2,
    label="M1 conic prescription",
)

plt.xlabel("M1 local radius r [mm]")
plt.ylabel("M1 local sag z [mm]")
plt.title("M1 ray intersections vs exact conic surface")
plt.legend()
plt.grid()
plt.show()


In [ ]:
m1_rmse = np.sqrt(np.nanmean(m1_df["z_residual_mm"]**2))

print(f"M1 local conic residual RMSE: {m1_rmse:.6e} mm")
print()
print(m1_df["z_residual_mm"].describe())



## Overlay analytic surfaces and saved rays in PCS

The analytic surfaces use their full meshes, while only a representative subset of the 10,000 ray paths is shown.


In [ ]:
def set_axes_to_data_aspect(ax, xyz):
    """Use physical x:y:z spans as the 3-D box aspect ratio."""
    xyz = np.asarray(xyz, dtype=float)
    good = np.all(np.isfinite(xyz), axis=1)

    if not np.any(good):
        return

    xyz = xyz[good]
    spans = np.ptp(xyz, axis=0)
    spans[spans == 0] = 1.0
    ax.set_box_aspect(spans)


fig = plt.figure(figsize=(13, 10))
ax = fig.add_subplot(111, projection="3d")

for name, surface in surfaces_pcs.items():
    ax.plot_surface(
        surface["X_pcs"],
        surface["Y_pcs"],
        surface["Z_pcs"],
        linewidth=0,
        alpha=0.35,
    )

    pts = surface["pcs_xyz"].reshape(-1, 3)
    good = np.all(np.isfinite(pts), axis=1)

    if np.any(good):
        center = np.mean(pts[good], axis=0)
        ax.text(center[0], center[1], center[2], name)

for ray in ray_indices:
    ax.plot(
        rays[:, ray, 0],
        rays[:, ray, 1],
        rays[:, ray, 2],
        linewidth=1.0,
        alpha=0.65,
    )

ax.set_xlabel("PCS x [mm]")
ax.set_ylabel("PCS y [mm]")
ax.set_zlabel("PCS z [mm]")

all_xyz = [rays[:, ray_indices, :].reshape(-1, 3)]

for surface in surfaces_pcs.values():
    all_xyz.append(surface["pcs_xyz"].reshape(-1, 3))

set_axes_to_data_aspect(
    ax,
    np.concatenate(all_xyz, axis=0),
)

plt.show()



## Inspect one surface against its ray hits

Useful whenever an individual surface looks unexpected in the full telescope plot.


In [ ]:
SURFACE_TO_INSPECT = "M3"

surface = surfaces_pcs[SURFACE_TO_INSPECT]

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

ax.plot_surface(
    surface["X_pcs"],
    surface["Y_pcs"],
    surface["Z_pcs"],
    linewidth=0,
    alpha=0.7,
)

idx = surface_specs[SURFACE_TO_INSPECT]["trace_index"]
hits = history[idx].reshape(-1, 3)
good = np.all(np.isfinite(hits), axis=1)
hits = hits[good]

if len(hits):
    sample = np.linspace(
        0,
        len(hits) - 1,
        min(500, len(hits)),
        dtype=int,
    )

    ax.scatter(
        hits[sample, 0],
        hits[sample, 1],
        hits[sample, 2],
        s=5,
    )

ax.set_xlabel("PCS x [mm]")
ax.set_ylabel("PCS y [mm]")
ax.set_zlabel("PCS z [mm]")

plt.show()


In [ ]:
def build_fpa_transform():
    """
    Same FPA transform used by PSFSim romantrace.py before intersecting the FPA.
    Local FPA coordinates -> PCS coordinates.
    """
    return build_transform_matrix(
        xde=866.9584811995454,
        yde=-1501.61613749036,
        zde=-407.5050041451383,
        ade=-62.41145131632292,
        bde=-27.09897706981732,
        cde=13.3889006733882,
    )


def get_fpa_sca_centers():
    """
    SCA centers in the local FPA coordinate system, in mm.
    These are the xfpa and yfpa arrays from wfi_coordinate_transformations.py.
    """
    xfpa = np.array([
        -22.14, -22.29, -22.44,
        -66.42, -66.92, -67.42,
        -110.70, -111.48, -112.64,
         22.14,  22.29,  22.44,
         66.42,  66.92,  67.42,
        110.70, 111.48, 112.64,
    ], dtype=float)

    yfpa = np.array([
         12.15, -37.03, -82.06,
         20.90, -28.28, -73.06,
         42.20,  -6.98, -51.06,
         12.15, -37.03, -82.06,
         20.90, -28.28, -73.06,
         42.20,  -6.98, -51.06,
    ], dtype=float)

    return xfpa, yfpa


def make_sca_square_local(x_center, y_center, half_size=20.44):
    """
    Build one SCA outline in the local FPA plane (z=0), in mm.

    half_size = 20.44 mm  -> full side length = 40.88 mm
    """
    return np.array([
        [x_center - half_size, y_center - half_size, 0.0],
        [x_center + half_size, y_center - half_size, 0.0],
        [x_center + half_size, y_center + half_size, 0.0],
        [x_center - half_size, y_center + half_size, 0.0],
        [x_center - half_size, y_center - half_size, 0.0],  # close outline
    ], dtype=float)


def build_fpa_sca_polygons_pcs(rb, half_size=20.44):
    """
    Build all 18 SCA polygons, transformed from local FPA coordinates into PCS.
    Returns a dict keyed by SCA number.
    """
    TrFPA = build_fpa_transform()
    xfpa, yfpa = get_fpa_sca_centers()

    sca_polygons_pcs = {}
    sca_centers_pcs = {}

    for sca in range(1, 19):
        x0 = xfpa[sca - 1]
        y0 = yfpa[sca - 1]

        square_local = make_sca_square_local(x0, y0, half_size=half_size)
        square_pcs = surface_to_pcs(rb, TrFPA, square_local)

        center_local = np.array([[x0, y0, 0.0]], dtype=float)
        center_pcs = surface_to_pcs(rb, TrFPA, center_local)[0]

        sca_polygons_pcs[sca] = square_pcs
        sca_centers_pcs[sca] = center_pcs

    return sca_polygons_pcs, sca_centers_pcs

In [ ]:
fig = plt.figure(figsize=(13, 10))
ax = fig.add_subplot(111, projection="3d")

# analytic surfaces
for name, surface in surfaces_pcs.items():
    ax.plot_surface(
        surface["X_pcs"],
        surface["Y_pcs"],
        surface["Z_pcs"],
        linewidth=0,
        alpha=0.35,
    )

    pts = surface["pcs_xyz"].reshape(-1, 3)
    good = np.all(np.isfinite(pts), axis=1)

    if np.any(good):
        center = np.mean(pts[good], axis=0)
        ax.text(center[0], center[1], center[2], name)

# 18 SCA outlines on FPA
sca_polygons_pcs, sca_centers_pcs = build_fpa_sca_polygons_pcs(rrb, half_size=20.44)

for sca, poly in sca_polygons_pcs.items():

    # outline
    ax.plot(
        poly[:, 0],
        poly[:, 1],
        poly[:, 2],
        linewidth=1.2,
    )

    # optional faint fill
    patch = Poly3DCollection([poly[:4]], alpha=0.12)
    ax.add_collection3d(patch)

    # label
    center = sca_centers_pcs[sca]
    ax.text(
        center[0],
        center[1],
        center[2],
        f"SCA{sca}",
        fontsize=7,
    )


# ray plotting
for ray in ray_indices:
    ax.plot(
        rays[:, ray, 0],
        rays[:, ray, 1],
        rays[:, ray, 2],
        linewidth=1.0,
        alpha=0.65,
    )


ax.set_xlabel("PCS x [mm]")
ax.set_ylabel("PCS y [mm]")
ax.set_zlabel("PCS z [mm]")

all_xyz = [rays[:, ray_indices, :].reshape(-1, 3)]

for surface in surfaces_pcs.values():
    all_xyz.append(surface["pcs_xyz"].reshape(-1, 3))

for poly in sca_polygons_pcs.values():
    all_xyz.append(poly.reshape(-1, 3))

set_axes_to_data_aspect(
    ax,
    np.concatenate(all_xyz, axis=0),
)

plt.show()